<a href="https://colab.research.google.com/github/LeMaterial/lematerial-llm-synthesis/blob/main/examples/notebooks/tutorials/06_customizing_the_ontology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 6 — Customising the ontology

`GeneralSynthesisOntology` decides what the pipeline extracts. Every synthesis
extractor writes into it, every judge reads out of it, and the published dataset
is stored in its shape. When your domain needs something it does not capture —
a sintering atmosphere, a lithium source, a measured surface area — this is the
file you change.

## What you'll learn

1. How the ontology is built, and how DSPy turns it into a prompt
2. **Recipe A** — add a field (the common case, one edit)
3. **Recipe B** — add a value to a closed enum, and find every prompt that
   repeats the list
4. **Recipe C** — use a completely different schema, and why
   `DspySynthesisExtractor` will not accept one

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- **Runtime:** ~10 min. **Cost:** free — this tutorial inspects and validates
  schemas, and makes no LLM calls.

## Setup — local or Colab

This notebook runs unchanged in two places:

- **Locally**, from a clone of the repository (`uv sync && uv pip install -e .`),
  with your API keys in the `.env` file at the repository root.
- **On [Google Colab](https://colab.research.google.com)** — click the badge at
  the top. The cell below clones the repository and installs it, which takes a
  few minutes the first time, then reads your keys from Colab's **secret
  manager**: open the 🔑 icon in the left sidebar, add one secret per key
  (`GEMINI_API_KEY`, `HF_TOKEN`, …) and switch *Notebook access* on for each.

Either way the keys land in `os.environ` and nothing else in the notebook
changes — no key is ever passed as a function argument, so none of them can end
up in the notebook's output or in git.

> If an import fails immediately after the setup cell on Colab, use
> **Runtime → Restart session** and run it again: the clone is cached, so the
> second run is quick.


In [ ]:
# --- Setup: this cell is the only difference between local and Colab -----
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Every key the project knows about. A tutorial only needs a subset; whichever
# ones are missing are reported by the key check further down.
API_KEY_NAMES = (
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "MISTRAL_API_KEY",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
    "HF_TOKEN",
)

if IN_COLAB:
    REPO_URL = "https://github.com/LeMaterial/lematerial-llm-synthesis.git"
    REPO_ROOT = Path("/content/lematerial-llm-synthesis")

    if not REPO_ROOT.exists():
        print("Cloning the repository ...")
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    print("Installing llm-synthesis (a few minutes on the first run) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_ROOT)],
        check=True,
    )

    # Colab keeps secrets outside the notebook, so they cannot leak into its
    # output: add them under the key icon in the left sidebar.
    from google.colab import userdata

    for name in API_KEY_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set, or notebook access not granted - reported below
    KEY_SOURCE = "Colab secrets"
else:
    from dotenv import find_dotenv, load_dotenv

    def find_repo_root(start: Path | None = None) -> Path:
        """Walk up from `start` (default: cwd) until a directory has pyproject.toml."""
        here = (start or Path.cwd()).resolve()
        for candidate in (here, *here.parents):
            if (candidate / "pyproject.toml").exists():
                return candidate
        raise RuntimeError(f"No pyproject.toml found above {here}")

    REPO_ROOT = find_repo_root()
    # find_dotenv walks up from the working directory, so this works whether you
    # started Jupyter at the repo root or inside this folder.
    env_path = find_dotenv(usecwd=True)
    load_dotenv(env_path, override=True)
    KEY_SOURCE = env_path or "no .env found"

print(f"environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f"repo root:   {REPO_ROOT}")
print(f"API keys:    {KEY_SOURCE}")


## Step 1 — What the model actually sees

The ontology is plain Pydantic. DSPy serialises the model's JSON schema —
including every `Field(description=...)` — into the prompt, which is why a
description is not documentation but *instruction*: it is the text the LLM reads
when deciding what to put in that field.

In [ ]:
from llm_synthesis.models.ontologies import GeneralSynthesisOntology

print("Top-level fields:\n")
for name, field in GeneralSynthesisOntology.model_fields.items():
    required = "required" if field.is_required() else "optional"
    description = (field.description or "")[:70]
    print(f"  {name:<22} {required:<9} {description}")

In [ ]:
import json

schema = GeneralSynthesisOntology.model_json_schema()

# The nested models the schema pulls in: Material, ProcessStep, Conditions, ...
print("Nested definitions:", list(schema.get("$defs", {})))

# This is (close to) what the LLM is asked to fill in.
print("\nConditions sub-schema, as the model sees it:")
print(json.dumps(schema["$defs"]["Conditions"], indent=2)[:900])

## Recipe A — Add a field

Say your papers report a **calcination atmosphere flow rate** and you want it
kept. Edit `src/llm_synthesis/models/ontologies/general.py` and add one field to
the relevant model — `Conditions` here:

```python
class Conditions(BaseModel):
    ...
    gas_flow_rate: float | None = Field(
        default=None,
        description=(
            "Gas flow rate during this step in mL/min. Null when the paper "
            "does not report one."
        ),
    )
    gas_flow_unit: str | None = Field(
        default=None,
        description="Unit of gas_flow_rate as written in the paper, e.g. 'mL/min'.",
    )
```

That is the entire change. No extractor, prompt or pipeline code needs
touching — DSPy re-serialises the schema on every call.

**Three rules that keep this safe:**

1. **Make it optional** (`| None` with `default=None`). Required fields make the
   model invent values, and break every previously stored record.
2. **Say what "absent" means** in the description. Without it, models write
   `0` or `"unknown"` instead of `null`.
3. **Keep units in their own field** rather than baking them into a number.

The cell below simulates the edit with a subclass so you can see the effect on
the prompt before touching the file.

In [ ]:
from pydantic import BaseModel, Field

from llm_synthesis.models.ontologies.general import Conditions


class ConditionsWithFlow(Conditions):
    """`Conditions` plus a gas flow rate - a preview of the edit above."""

    gas_flow_rate: float | None = Field(
        default=None,
        description=(
            "Gas flow rate during this step in mL/min. Null when the paper "
            "does not report one."
        ),
    )
    gas_flow_unit: str | None = Field(
        default=None,
        description="Unit of gas_flow_rate as written in the paper, e.g. 'mL/min'.",
    )


before = set(Conditions.model_fields)
after = set(ConditionsWithFlow.model_fields)
print("New fields the LLM would be asked for:", sorted(after - before))

print("\nHow they appear in the prompt schema:")
properties = ConditionsWithFlow.model_json_schema()["properties"]
for name in sorted(after - before):
    print(f"  {name}: {properties[name].get('description')}")

In [ ]:
# Old records still load: the new fields simply default to None.
legacy = {
    "temperature": 450.0,
    "temp_unit": "C",
    "duration": 2.0,
    "time_unit": "h",
}
restored = ConditionsWithFlow.model_validate(legacy)
print(f"temperature={restored.temperature} {restored.temp_unit}")
print(f"gas_flow_rate={restored.gas_flow_rate} (defaulted, nothing broke)")

## Recipe B — Add a value to a closed enum

`target_compound_type` and `synthesis_method` are `Literal` enums, not free
text. That is what makes the published dataset groupable — but it means a value
the ontology does not know about is a **validation error**, not a new category.

Adding one is two edits and a search:

1. Add the string to the `Literal[...]` in
   `src/llm_synthesis/models/ontologies/general.py`.
2. Add it to every prompt that repeats the list — because the system prompts
   spell out the allowed values, a schema that has drifted from its prompts
   makes the model emit values that then fail validation.

There is no single source that generates those prompt copies, so the check has
to be mechanical. The next cell does it for the enums as they stand today.

In [ ]:
from pathlib import Path
from typing import get_args

# Files that repeat the enum values in prose for the LLM.
PROMPT_FILES = [
    REPO_ROOT / "config" / "cli.yaml",
    REPO_ROOT
    / "examples"
    / "system_prompts"
    / "synthesis_extraction"
    / "default.txt",
]

ENUM_FIELDS = ["synthesis_method", "target_compound_type"]


def enum_values(field_name):
    """The Literal values a field currently accepts."""
    return get_args(
        GeneralSynthesisOntology.model_fields[field_name].annotation
    )


def check_prompt_sync(prompt_files, enum_fields):
    """Report enum values that the ontology allows but a prompt never mentions."""
    for path in prompt_files:
        if not path.exists():
            print(f"{path.relative_to(REPO_ROOT)}: NOT FOUND")
            continue
        text = path.read_text()
        print(f"\n{path.relative_to(REPO_ROOT)}")
        for field_name in enum_fields:
            missing = [
                value
                for value in enum_values(field_name)
                if f"'{value}'" not in text and f'"{value}"' not in text
            ]
            status = ", ".join(missing) if missing else "in sync"
            print(f"  {field_name:<22} {status}")


for field_name in ENUM_FIELDS:
    print(f"{field_name}: {len(enum_values(field_name))} allowed values")

check_prompt_sync(PROMPT_FILES, ENUM_FIELDS)

> **If that cell reported missing values, it found real drift** — those are
> values the schema accepts but the prompt never offers, so the model will
> rarely produce them. Fixing it means pasting the value into the prompt list.
>
> The system prompts embedded in `examples/scripts/` and in the tutorial
> notebooks carry their own copies of these lists. To find every one of them:
>
> ```bash
> grep -rl "incipient wetness impregnation" --include="*.py" --include="*.yaml" \
>     --include="*.txt" --include="*.ipynb" .
> ```

In [ ]:
# What a stale enum costs you: the model returns a plausible-but-unknown value
# and the whole record fails validation.
import pydantic

try:
    GeneralSynthesisOntology.model_validate(
        {
            "target_compound": "LiFePO4",
            "target_compound_type": "battery materials",  # not in the Literal
            "synthesis_method": "solid-state",
        }
    )
except pydantic.ValidationError as error:
    print(f"Rejected, as designed:\n  {error.errors()[0]['msg'][:160]}")

## Recipe C — A schema of your own

Sometimes you do not want `GeneralSynthesisOntology` at all — you want a small,
domain-specific record. You can do that, with one constraint to know about
first.

`DspySynthesisExtractor` validates its signature on construction and **requires
the output field to be exactly `GeneralSynthesisOntology`** — not a subclass, not
a similar model. So a custom schema means writing a small extractor module of
your own. It is about ten lines.

In [ ]:
import dspy


class CathodeStep(BaseModel):
    step_number: int = Field(description="1-based position of this step.")
    action: str = Field(description="Short verb phrase describing the step.")


class CathodeSynthesis(BaseModel):
    """A deliberately small, battery-specific alternative to the general ontology."""

    target_compound: str = Field(description="Cathode material synthesised.")
    lithium_source: str | None = Field(
        default=None,
        description="Lithium precursor, e.g. 'Li2CO3'. Null if not reported.",
    )
    calcination_temperature_c: float | None = Field(
        default=None,
        description="Final calcination temperature in Celsius. Null if absent.",
    )
    steps: list[CathodeStep] = Field(
        default_factory=list, description="Ordered synthesis steps."
    )


cathode_signature = dspy.make_signature(
    signature_name="ExtractCathodeSynthesis",
    instructions=(
        "Extract the cathode synthesis procedure for the specified material. "
        "Report only what the paper states; use null for anything absent."
    ),
    signature={
        "paper_text": (str, dspy.InputField(description="Full paper text.")),
        "material_name": (
            str,
            dspy.InputField(
                description="Material to extract the synthesis for."
            ),
        ),
        "cathode_synthesis": (
            CathodeSynthesis,
            dspy.OutputField(description="Structured cathode synthesis."),
        ),
    },
)

print("Signature inputs :", list(cathode_signature.input_fields))
print("Signature output :", list(cathode_signature.output_fields))
print(
    "Prompt schema    :",
    list(CathodeSynthesis.model_json_schema()["properties"]),
)

In [ ]:
# The library extractor refuses this signature - by design.
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
)
import os

from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.llms import SystemPrefixedLM

# No LLM call is made in this tutorial - we only need an LM object to hand to
# the extractor. Flip USE_OPENROUTER to route through OpenRouter instead of a
# direct provider key; the same two-branch helper appears in every tutorial
# that does call a model.
USE_OPENROUTER = False

if USE_OPENROUTER:
    lm = SystemPrefixedLM(
        "",
        "openrouter/google/gemini-3-flash-preview",
        api_base="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        temperature=0.0,
    )
else:
    lm = get_llm_from_name(
        "gemini-3.0-flash", model_kwargs={"temperature": 0.0}
    )

try:
    DspySynthesisExtractor(signature=cathode_signature, lm=lm)
except ValueError as error:
    print(f"DspySynthesisExtractor says: {error}")

In [ ]:
class CathodeSynthesisExtractor(dspy.Module):
    """Minimal stand-in for DspySynthesisExtractor with a custom output schema.

    The library version adds temperature-escalation retries, bare-JSON recovery
    and a fallback record; copy those from
    `src/llm_synthesis/transformers/synthesis_extraction/dspy_synthesis_extraction.py`
    if you need production robustness.
    """

    def __init__(self, signature: type[dspy.Signature], lm: dspy.LM):
        super().__init__()
        self.signature = signature
        self.lm = lm

    def forward(self, paper_text: str, material_name: str) -> CathodeSynthesis:
        with dspy.settings.context(
            lm=self.lm, adapter=dspy.adapters.JSONAdapter()
        ):
            prediction = dspy.Predict(self.signature)(
                paper_text=paper_text, material_name=material_name
            )
        return prediction.cathode_synthesis


extractor = CathodeSynthesisExtractor(cathode_signature, lm)
print("Custom extractor ready:", type(extractor).__name__)

# Uncomment to run it for real (costs one LLM call and needs GEMINI_API_KEY):
# result = extractor.forward(paper_text=DEMO_PAPER, material_name="LiFePO4")
# print(result.model_dump_json(indent=2))

### What you give up

A custom schema is a fork of the ecosystem, not just of the model:

| You lose | Because |
|----------|---------|
| `DspyGeneralSynthesisJudge` | It reads `GeneralSynthesisOntology` fields by name |
| `SynthesisPerformancePipeline` and the CLI | Both are typed against the general ontology |
| Comparability with the published dataset | Different schema, different columns |

So prefer **Recipe A** — add optional fields to the general ontology — unless
your record genuinely has nothing in common with it. Optional fields cost
nothing to models that never fill them in, and everything downstream keeps
working.

## Checklist for a schema change

1. Add the field or enum value in
   `src/llm_synthesis/models/ontologies/general.py`
2. Write a `Field(description=...)` that tells the model when to leave it null
3. Re-run the prompt-sync cell above; update every prompt copy it flags
4. Re-validate a stored record (`model_validate`) to confirm old data still loads
5. Extract one paper and read the output before running a batch
6. Note that judge scores are not comparable across schema versions — re-judge
   rather than mixing

## What's next

- **[Tutorial 4 — Synthesis + performance from a paper](04_extracting_synthesis_and_performance.ipynb)**:
  run your changed schema on a real paper.
- **[Tutorial 5 — Evaluating extraction quality](05_evaluating_extraction_quality.ipynb)**:
  check whether the new field is actually being filled in correctly.
- **[Tutorial 3 — Batch extraction with the CLI](03_batch_extraction_with_the_cli.ipynb)**:
  `"prompts.synthesis_instructions=..."` steers extraction without a schema
  change at all — try that first when the schema is already sufficient.